# melvil on AG News

End-to-end walkthrough: labeled CSV → optimized, versioned classifier prompt → evaluation report.

Costs well under $1 with the default models (`budget="light"`). Set `OPENROUTER_API_KEY` (or `OPENAI_API_KEY` and the `openai/...` model ids) before running.

In [ ]:
import csv
import logging
import pathlib

import melvil as mv

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
mv.__version__

## 1. Data

Any CSV with `text,label` columns works. Here we synthesize one from HuggingFace AG News.

In [ ]:
csv_path = pathlib.Path("agnews_small.csv")
if not csv_path.exists():
    rows = mv.load_hf("fancyzhx/ag_news", split="train", limit=1200)
    with open(csv_path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["text", "label"])
        w.writerows((e.text, e.label) for e in rows)

examples = mv.load_csv(csv_path)
train, dev = mv.train_dev_split(examples, dev_size=100, seed=0)
spec = mv.TaskSpec.from_examples("agnews-topics", examples)
spec.label_names

## 2. Configure and estimate cost

Everything is a plain dataclass. The estimate is an upper bound — print it before spending.

In [ ]:
cfg = mv.Config(task_model="openrouter/openai/gpt-4.1-mini",
                reflection_model="openrouter/openai/gpt-4.1",
                budget="light")
print(mv.estimate_optimize_cost(spec, train, dev, cfg))

## 3. Optimize

`on_round` fires after every full dev evaluation — live progress with no extra machinery. `resume=True` makes the cell re-runnable (it picks up the finished artifact or continues an interrupted run).

In [ ]:
def show(info: mv.RoundInfo):
    print(f"round {info.round}: dev {info.dev_score:.3f} "
          f"(best {info.best_dev_score:.3f}) — ${info.cost_usd:.2f}, "
          f"top confusions: {info.top_confusions}")

artifact = mv.optimize(spec, train, dev, cfg, resume=True, on_round=show)
print(mv.report(artifact))

## 4. The deployable prompt

`render()` is exactly the prompt that was optimized — per-label definitions, boundary rules, mined exemplars, and a fixed output contract.

In [ ]:
print(artifact.render())

## 5. Evaluate on held-out data

`evaluate` returns accuracy, macro-F1, per-label precision/recall, the confusion matrix, and measured cost; `report` renders markdown. Pass a different `model=` for a transfer evaluation.

In [ ]:
test = mv.load_hf("fancyzhx/ag_news", split="test", limit=300)
rep = mv.evaluate(artifact, test)
print(mv.report(rep))

## 6. Version and compare

Artifacts are versioned JSON with lineage — save them next to your code, diff them across iterations.

In [ ]:
artifact.save("agnews_topics.v1.json")
loaded = mv.PromptArtifact.load("agnews_topics.v1.json")
assert loaded.render() == artifact.render()

# a second run with a different seed, recording lineage:
# child = mv.optimize(spec, train, dev, mv.Config(..., seed=1), parent=artifact)
# print(child.diff(artifact))